# 06 — Carteira ótima α* (FOC)

Desenvolve `funcao_foc` e `resolver_alpha_otimo` (matemática pura). **F6.**

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np
from scipy import optimize

## Desenvolvimento

A(s) função(ões)/classe(s) abaixo foi(ram) escrita(s) aqui e, após os testes, movida(s) para `app/nucleo.py`.

In [ ]:
_TOL_FALENCIA = 1e-12

# Bracket inicial do brentq (N=1), duplicado ate _MAX_EXPANSOES vezes.
_BRACKET_INICIAL = 20.0
_MAX_EXPANSOES = 8

def funcao_foc(alpha, R, rf, gamma):
    """G(α) = E[(R − rf·1) / R_p^γ], com R_p = rf + αᵀ(R − rf·1). Shape (N,). (F6)"""
    alpha = np.asarray(alpha, dtype=float)
    excesso = R - rf                                    # (n, N)
    R_p = np.maximum(rf + excesso @ alpha, _TOL_FALENCIA)  # (n,)
    pesos = R_p ** (-gamma)
    return (excesso.T @ pesos) / R.shape[0]


def _objetivo_J(alpha, R, rf, gamma):
    """J(α) = E[u(R_p(α))] — maximizado pelo solver (dJ/dα = G)."""
    excesso = R - rf
    R_p = np.maximum(rf + excesso @ alpha, _TOL_FALENCIA)
    if np.isclose(gamma, 1.0):
        u = np.log(R_p)
    else:
        u = R_p ** (1.0 - gamma) / (1.0 - gamma)
    return float(u.mean())


def resolver_alpha_otimo(R, rf, gamma, *, tol=1e-10, maxiter=200, alpha0=None):
    """Resolve a FOC G(α*)=0 maximizando J(α). (F6)

    **Fiel ao artigo: α ∈ ℝ^N, SEM restrição alguma** — venda a descoberto
    (α<0) e alavancagem (Σα>1) são sempre admitidas.
    N ≥ 2 → SLSQP irrestrito; N = 1 → brentq com bracket auto-expansível.

    α* finito existe porque J é côncava e G(α) é decrescente: basta que a
    amostra contenha algum cenário com R < rf e algum com R > rf. Se isso
    falhar — arbitragem na amostra — levanta ``RuntimeError`` em vez de
    devolver um valor de canto arbitrário.
    """
    R = np.asarray(R, dtype=float)
    N = R.shape[1]
    if alpha0 is None:
        alpha0 = np.full(N, 1.0 / N)

    if N >= 2:
        res = optimize.minimize(
            lambda a: -_objetivo_J(a, R, rf, gamma), np.asarray(alpha0, float),
            jac=lambda a: -funcao_foc(a, R, rf, gamma),
            method="SLSQP",
            options={"ftol": tol, "maxiter": int(maxiter), "disp": False})
        return res.x.copy()

    # N == 1 — brentq. G decresce em α: procura-se G(lo) > 0 > G(hi).
    g = lambda a: float(funcao_foc(np.array([a]), R, rf, gamma)[0])
    lo = -_BRACKET_INICIAL
    hi = _BRACKET_INICIAL
    for _ in range(_MAX_EXPANSOES):
        if g(lo) > 0 > g(hi):
            return np.array([optimize.brentq(g, lo, hi, xtol=tol)])
        lo *= 2.0
        hi *= 2.0
    raise RuntimeError(
        f"FOC sem troca de sinal em α ∈ [{lo / 2:.0f}, {hi / 2:.0f}]: não há α* "
        "finito. Verifique se a amostra de R contém cenários acima e abaixo de rf."
    )


**Teste** — resíduo da FOC ≈ 0 e aproximação de Merton (1 ativo).

In [3]:
rng = np.random.default_rng(0); r = rng.normal(0.008,0.04,300_000); rf, g = 0.003, 4.0
R = np.maximum(1+r,0).reshape(-1,1)
a = resolver_alpha_otimo(R, 1+rf, g); merton = (r.mean()-rf)/(g*r.var(ddof=1))
print('alpha*:', a, '| Merton:', merton, '| G(a*):', funcao_foc(a, R, 1+rf, g))
assert abs(funcao_foc(a,R,1+rf,g)[0]) < 1e-6 and abs(a[0]-merton)/merton < 0.05
print('F6 carteira_otima: PASSOU')

alpha*: [0.78278503] | Merton: 0.7815586515070274 | G(a*): [-4.0251121e-15]
F6 carteira_otima: PASSOU


✔ **Testado e aprovado — código movido para `app/nucleo.py`.**